In [1]:
import pandas as pd
from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

In [2]:
load_dotenv()

engine = create_engine(
    f"postgresql://{os.getenv('DB_DESTINATION_USER')}:{os.getenv('DB_DESTINATION_PASSWORD')}@"
    f"{os.getenv('DB_DESTINATION_HOST')}:{os.getenv('DB_DESTINATION_PORT')}/{os.getenv('DB_DESTINATION_NAME')}"
)

In [3]:
df = pd.read_sql("SELECT * FROM merged_flats", engine)

print("Дубликаты:", df.duplicated().sum())
print("Пропуски:\n", df.isnull().sum())
print("Описательная статистика:\n", df.describe())

Дубликаты: 0
Пропуски:
 flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64
Описательная статистика:
              flat_id    building_id          floor   kitchen_area  \
count  141362.000000  141362.000000  141362.000000  141362.000000   
mean    70680.500000   14053.665235       7.467346       9.001579   
std     40807.838714    6988.831066       5.717144       5.264076   
min         0.000000       1.000000       1.000000       0.000000   
25%     35340.250000    8535.250000       3.000000       6.100000   
50%     70680.500000   14332.000000       6.000000       8.800000   
75%    106020.750000   20475.000000     

In [4]:
print("Максимальная цена:", df['price'].max())
print("99-й перцентиль цены:", df['price'].quantile(0.99))
print("Максимальная площадь:", df['total_area'].max())
print("99-й перцентиль площади:", df['total_area'].quantile(0.99))
print("Максимальное кол-во комнат:", df['rooms'].max())

Максимальная цена: 9873737728.0
99-й перцентиль цены: 152000000.0
Максимальная площадь: 960.2999877929688
99-й перцентиль площади: 223.8999962768555
Максимальное кол-во комнат: 20


In [5]:
def remove_duplicates(df):
    return df.drop_duplicates()

def handle_missing_values(df):
    df = df.dropna(subset=['price', 'total_area', 'rooms', 'floor'])
    df['ceiling_height'] = df['ceiling_height'].fillna(df['ceiling_height'].median())
    df['kitchen_area'] = df['kitchen_area'].fillna(df['kitchen_area'].median())
    df['living_area'] = df['living_area'].fillna(df['living_area'].median())
    return df

def remove_outliers(df):
    df = df[
        (df['price'] > 0) & 
        (df['total_area'] > 0) & 
        (df['floor'] > 0) & 
        (df['rooms'] >= 0) &
        (df['rooms'] <= 10) &
        (df['price'] < df['price'].quantile(0.99)) &
        (df['total_area'] < df['total_area'].quantile(0.99))
    ]
    return df

def clean_data(df):
    df = remove_duplicates(df)
    df = handle_missing_values(df)
    df = remove_outliers(df)
    return df

In [6]:
df_clean = clean_data(df)
print("Итоговый размер:", df_clean.shape)
print("Пропуски после очистки:\n", df_clean.isnull().sum())
df_clean.to_sql('cleaned_flats_test', engine, if_exists='replace', index=False)

Итоговый размер: (139328, 18)
Пропуски после очистки:
 flat_id              0
building_id          0
floor                0
kitchen_area         0
living_area          0
rooms                0
is_apartment         0
studio               0
total_area           0
price                0
build_year           0
building_type_int    0
latitude             0
longitude            0
ceiling_height       0
flats_count          0
floors_total         0
has_elevator         0
dtype: int64


328